# Fooocus_extend – Google Colab

Diese Version ist für die einfache Weitergabe in Schulungen und Workshops gedacht.

## Ablauf
1. Oben in Colab: **Laufzeit → Laufzeittyp ändern → GPU**
2. **Zelle 1** einmal ausführen.
3. Falls Colab automatisch neu startet: kurz warten, bis die Verbindung wieder da ist.
4. Danach **Zelle 2** ausführen.
5. Den angezeigten `gradio.live`-Link öffnen.

## Datenhaltung
- **Notebook:** dauerhaft in Google Drive oder GitHub.
- **Programmumgebung:** liegt unter `/content` in der temporären Colab-VM.
- **Bilder:** standardmäßig nur temporär. Nur wenn `GoogleDrive_output = True` gesetzt wird, werden Ausgaben dauerhaft in `MyDrive/outputs` gespeichert.

**Sicherheit:** Google Drive nur bei Bedarf verbinden. Sobald Drive gemountet ist, kann Notebook-Code auf die dort erreichbaren Dateien zugreifen. Keine Passwörter, Tokens, API-Keys oder vertraulichen Inhalte in Codezellen, Ausgaben oder ein öffentliches GitHub-Repository schreiben.

Nach der Übung: **Laufzeit → Laufzeit trennen und löschen**.


In [ ]:
# @title 1. Vorbereitung / Abhängigkeiten
import sys
import subprocess
import IPython
from importlib.metadata import version, PackageNotFoundError

required_packages = {
    "nvidia-cudnn-cu12": "9.1.0.70",
    "pygit2": "1.15.1",
    "numpy": "1.26.4",
}

def installed_version(package):
    try:
        return version(package)
    except PackageNotFoundError:
        return None

to_install = []

for package, wanted_version in required_packages.items():
    current = installed_version(package)
    if current != wanted_version:
        print(f"{package}: {current or 'nicht installiert'} → {wanted_version}")
        to_install.append(f"{package}=={wanted_version}")

if to_install:
    print("\nBenötigte Pakete werden angepasst ...")
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "--quiet", *to_install
    ])
    print("\n✓ Pakete erfolgreich installiert.")
    print("Colab wird jetzt einmal neu gestartet.")
    print("Danach bitte Zelle 2 ausführen.")

    IPython.Application.instance().kernel.do_shutdown(restart=True)
else:
    print("✓ Alle benötigten Pakete sind bereits korrekt installiert.")
    print("Du kannst direkt mit Zelle 2 weitermachen.")


In [ ]:
# @title 2. Fooocus_extend starten

import os
import sys
import shutil
import subprocess
from pathlib import Path
from importlib.metadata import version

Fooocus_Profile = "realistic" #@param ["default", "realistic", "anime"]
Fooocus_Theme = "dark" #@param ["dark", "light"]
Tunnel = "gradio" #@param ["gradio", "cloudflared"]
Memory_patch = True #@param {type:"boolean"}
GoogleDrive_output = False #@param {type:"boolean"}

REPO_URL = "https://github.com/shaitanzx/Fooocus_extend.git"
REPO_DIR = Path("/content/Fooocus_extend")
OUTPUT_DIR = Path("/content/drive/MyDrive/outputs")
PORT = "7865"

required = {
    "nvidia-cudnn-cu12": "9.1.0.70",
    "pygit2": "1.15.1",
    "numpy": "1.26.4",
}

wrong = []
for package, wanted in required.items():
    try:
        current = version(package)
    except Exception:
        current = None
    if current != wanted:
        wrong.append(f"{package}: {current or 'fehlt'} → {wanted}")

if wrong:
    raise RuntimeError(
        "Die Vorbereitung ist noch nicht abgeschlossen.\n"
        "Bitte zuerst Zelle 1 ausführen und den Colab-Neustart abwarten.\n\n"
        + "\n".join(wrong)
    )

print("✓ Abhängigkeiten stimmen.")

gpu_check = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
    capture_output=True,
    text=True
)

if gpu_check.returncode != 0:
    raise RuntimeError(
        "Keine NVIDIA-GPU erkannt.\n\n"
        "Bitte in Colab unter Laufzeit → Laufzeittyp ändern → GPU auswählen."
    )

print("✓ GPU:", gpu_check.stdout.strip())

import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "Die GPU ist vorhanden, aber PyTorch erkennt CUDA nicht.\n"
        "Bitte die Colab-Laufzeit neu starten und Zelle 2 erneut ausführen."
    )

print("✓ CUDA:", torch.cuda.get_device_name(0))

subprocess.run(["pkill", "-f", "entry_with_update.py"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
subprocess.run(["pkill", "-f", "cloudflared tunnel"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

if not REPO_DIR.exists():
    print("\nFooocus_extend wird installiert ...")
    subprocess.check_call(["git", "clone", REPO_URL, str(REPO_DIR)])
else:
    if not (REPO_DIR / ".git").exists():
        raise RuntimeError("/content/Fooocus_extend existiert, ist aber kein gültiges Git-Repository.")
    print("\nFooocus_extend ist bereits vorhanden – Programmcode wird aktualisiert.")
    subprocess.check_call(["git", "-C", str(REPO_DIR), "fetch", "origin", "main"])
    subprocess.check_call(["git", "-C", str(REPO_DIR), "reset", "--hard", "origin/main"])

print("✓ Fooocus_extend ist aktuell.")

output_args = []

if GoogleDrive_output:
    from google.colab import drive
    if not Path("/content/drive/MyDrive").exists():
        print("\nGoogle Drive wird verbunden ...")
        drive.mount("/content/drive", force_remount=False)
    else:
        print("✓ Google Drive ist bereits verbunden.")
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    output_args = ["--output-path", str(OUTPUT_DIR)]
    print("✓ Ausgabeordner:", OUTPUT_DIR)

if Tunnel == "cloudflared":
    print("\nCloudflared wird vorbereitet ...")
    if shutil.which("cloudflared") is None:
        deb_file = "/tmp/cloudflared-linux-amd64.deb"
        subprocess.check_call([
            "wget", "-q", "-O", deb_file,
            "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb"
        ])
        subprocess.check_call(["dpkg", "-i", deb_file])
    subprocess.check_call([sys.executable, str(REPO_DIR / "patcher_tunel.py")])

args = [sys.executable, "entry_with_update.py", "--port", PORT]

if Fooocus_Profile == "realistic":
    args += ["--preset", "realistic"]
elif Fooocus_Profile == "anime":
    args += ["--preset", "anime"]

if Fooocus_Theme == "dark":
    args += ["--theme", "dark"]

if Tunnel == "gradio":
    args += ["--share"]

if Memory_patch:
    args += ["--always-high-vram", "--all-in-fp16"]

args += output_args
os.chdir(REPO_DIR)

print("\n" + "=" * 60)
print("FOOOCUS_EXTEND WIRD GESTARTET")
print("=" * 60)
print("Profil:              ", Fooocus_Profile)
print("Theme:               ", Fooocus_Theme)
print("Tunnel:              ", Tunnel)
print("Memory Patch:        ", Memory_patch)
print("Google Drive Output: ", GoogleDrive_output)
print("GPU:                 ", torch.cuda.get_device_name(0))
print("=" * 60 + "\n")

subprocess.run(args, check=True)


## Speichern und weitergeben

Das Notebook selbst kann dauerhaft in **Google Drive** gespeichert oder über **GitHub** bereitgestellt werden.

Für ein öffentliches Schulungs-Repository sollten nur diese Dateien enthalten sein:
- `Fooocus_Extend_VSW_Colab.ipynb`
- `README.md`
- ggf. `LICENSE`

**Nicht hochladen:** generierte Bilder, Referenzbilder, Modelle/LoRAs, Zugangsdaten, API-Keys oder andere personenbezogene bzw. vertrauliche Dateien.

Vor dem Veröffentlichen empfiehlt sich in Colab: **Bearbeiten → Notebook-Einstellungen → Ausgabe von Codezellen beim Speichern auslassen**.

So wird der Programmcode geteilt, nicht die temporäre Colab-VM.
